In [7]:
import chromadb 
client = chromadb.PersistentClient(
    path = "./chroma_db"
)

collection = client.get_or_create_collection("my_collection")


In [3]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

c:\Users\Intro\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2862.95it/s]


In [6]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

import fitz 
read_pdf = fitz.open(r"C:\Users\Intro\Downloads\constitutionofindia.pdf")

storage = ""
for i in read_pdf:
    storage += i.get_text()
    
len(storage)

sliding_window = []
for i in range(0, len(storage), 1300):
    chunk = storage[i: i + 1500]
    sliding_window.append(chunk)

len(sliding_window)

embeddings = model.encode(sliding_window)
print(embeddings.shape)

collection.add(
    documents = sliding_window,
    embeddings = embeddings, 
    metadatas = [{"source": "constitution.pdf", "chunk_id": i} for i in range(len(sliding_window))],
    ids = [f"chunks_{i}" for i in range(len(sliding_window))]
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1856.86it/s]


(669, 384)


In [ ]:
from groq import Groq 

client = Groq(api_key = "your api key")


In [11]:
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_or_create_collection("my_collection")

def ask(query):
    query_embedding = model.encode(query)
    results = collection.query(
        query_embeddings=query_embedding,
        n_results = 2
    )
    prompt = f"context: {results['documents']} \nQuestion {query} \n Answer based only on the context above, if you cant find the information then reply with the message i cannot find the info"
    response_rag = client.chat.completions.create(
        model = "llama-3.1-8b-instant",
        messages = [{"role":"user", "content":prompt}],
        max_tokens = 2000
    )
    return response_rag.choices[0].message.content
    
print(ask("what are the fundamental rights of indian citizen"))


Based on the provided context, I can identify some of the fundamental rights of Indian citizens. 

Here's the list of fundamental rights:

- Continuance of the rights of citizenship (Article 10)
- Right to Equality (Article 14-18)
  - Article 14: Equality before law
  - Article 15: Prohibition of discrimination on grounds of religion, race, caste, sex or place of birth
  - Article 16: Equality of opportunity in matters of public employment
  - Article 17: Abolition of Untouchability
  - Article 18: Abolition of titles
- Right to Freedom (Article 19-22)
  - Article 19: Protection of certain rights regarding freedom of speech, etc.
  - Article 20: Protection in respect of conviction for offences
  - Article 21: Protection of life and personal liberty
  - Article 21A: Right to education
  - Article 22: Protection against arrest and detention in certain cases
- Right against Exploitation (Article 23-24)
  - Article 23: Prohibition of traffic in human beings and forced labour
  - Article 24